# SIDM ABCD signal-region counting limits

Builds [Combine](https://cms-analysis.github.io/HiggsAnalysis-CombinedLimit/) datacards from
the counts in the ABCD **signal region** (region A) of the merged coffea outputs, one card per
signal point per channel, and writes them to `datacards/`.

Once the cards exist, run Combine over them with

```bash
python sidm/scripts/run_combine_limits.py -j 8
```

which writes `limits/limits.csv` and `limits/limits.json` back into this folder. The last
section of this notebook reads that file back and plots the expected limits.

### What goes into a card

The merged coffea files store each observable as a `Hist` with axes
`(channel, <observable>, abcd_region)`, where `abcd_region` is an `IntCategory` with
`0 = A`. So the SR count for a sample is that histogram sliced at `abcd_region = 0` for the SR
channel and summed over the observable axis, exactly as in

```python
h = output['out']['QCD_Pt300To470']['hists']['abcd_2mu2e_mulj_pt']
h[{"channel": "test_SR_2mu2e_spread_cosAlpha_mu_veto", "abcd_region": hist.loc(0)}].sum()
```

with two adjustments, both handled by `datacard_tools.sr_yield`:

* the sum uses `flow=True`, because the observable axes overflow at the few-percent level.
  With flow included the SR sum reproduces the final row of the corresponding cutflow exactly;
* the histograms are already scaled to `lumi * xs` by `sidm_processor.postprocess`, so these
  sums are yields, not raw entries, and the `Weight` storage gives their MC statistical
  uncertainty for free.

Signal samples have no entry in `configs/cross_sections.yaml`, so `utilities.get_xs` falls back
to **1 fb**. Combine's signal strength `r` is therefore a multiplier on 1 fb, i.e. the limit on
`r` *is* the limit on the signal cross section in fb.

In [ ]:
import sys
import os
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# local
sys.path.insert(1, os.path.join(os.getcwd(), '../../..'))
sys.path.insert(1, os.getcwd())
import datacard_tools
from sidm.tools import utilities
importlib.reload(datacard_tools)

utilities.set_plot_style()
%matplotlib inline

STUDY_DIR = Path(os.getcwd())
DATACARD_DIR = STUDY_DIR / "datacards"
LIMIT_DIR = STUDY_DIR / "limits"
CACHE = STUDY_DIR / "sr_yields.pkl"

print("backgrounds:", datacard_tools.BKG_DIR)
print("signal     :", datacard_tools.SIGNAL_DIR)
for name, ch in datacard_tools.CHANNELS.items():
    print(f"{name:10s} <- {ch.selection}  (counted with {ch.hist_name})")

## 1. Load the coffea files and extract the signal-region yields

Reading all 18 background and 120 signal files takes about a minute, so the result is cached in `sr_yields.pkl`. Set `reload = True` to re-read the coffea files.

In [ ]:
reload = False

def progress(i, n, name):
    if i % 20 == 0 or i == n - 1:
        print(f"  [{i + 1}/{n}] {name}", flush=True)

if not reload and CACHE.exists():
    cached = pd.read_pickle(CACHE)
    bkg_yields, signal_yields = cached["bkg"], cached["signal"]
    print(f"loaded cached yields from {CACHE.name}")
else:
    print("backgrounds:")
    bkg_yields = datacard_tools.collect_yields(datacard_tools.BKG_DIR, progress=progress)
    print("signal:")
    signal_yields = datacard_tools.collect_yields(datacard_tools.SIGNAL_DIR, progress=progress)
    pd.to_pickle({"bkg": bkg_yields, "signal": signal_yields}, CACHE)
    print(f"cached to {CACHE.name}")

print(f"\n{len(bkg_yields)} background samples, {len(signal_yields)} signal points")

### Background yields per sample

The uncertainties here are MC statistical only. Most samples contribute nothing at all to the SR, and the ones that do come from one or two raw simulated events, so the relative uncertainties are enormous — see the note in section 3.

In [ ]:
rows = []
for sample, per_channel in sorted(bkg_yields.items()):
    row = {"sample": sample, "group": datacard_tools.bkg_group(sample)}
    for ch, y in per_channel.items():
        row[ch] = y.value
        row[f"{ch}_err"] = y.error
    rows.append(row)

bkg_table = pd.DataFrame(rows).set_index("sample")
bkg_table.style.format("{:.4f}", subset=[c for c in bkg_table.columns if c != "group"])

In [ ]:
bkg_grouped = datacard_tools.group_backgrounds(bkg_yields)

rows = []
for ch, processes in bkg_grouped.items():
    for process, y in sorted(processes.items()):
        rows.append({
            "channel": ch, "process": process, "yield": y.value,
            "mc_stat_err": y.error, "rel_err": y.rel_error,
            # effective raw entries behind the yield, which is what the gmN
            # nuisance in the datacard is built from
            "n_raw_eff": round(1 / y.rel_error**2) if y.rel_error > 0 else 0,
        })
    total = sum((y for y in processes.values()), datacard_tools.Yield(0.0, 0.0))
    rows.append({"channel": ch, "process": "TOTAL", "yield": total.value,
                 "mc_stat_err": total.error, "rel_err": total.rel_error, "n_raw_eff": np.nan})

grouped_table = pd.DataFrame(rows)
grouped_table

### Signal yields

Each signal point is counted in the channel targeting its final state: `2Mu2E` samples in `SR_2mu2e`, `4Mu` samples in `SR_4mu`. The cross-final-state yields are at the 1e-4 level and carry no sensitivity, so they are not used.

In [ ]:
rows = []
for sample, per_channel in signal_yields.items():
    info = datacard_tools.parse_signal_name(sample)
    if info is None:
        print(f"skipping unparsable signal name: {sample}")
        continue
    channel = next(name for name, ch in datacard_tools.CHANNELS.items()
                   if ch.signal_prefix == info["final_state"])
    y = per_channel[channel]
    rows.append({"signal": sample, "channel": channel, **info,
                 "yield": y.value, "mc_stat_err": y.error, "rel_err": y.rel_error})

signal_table = pd.DataFrame(rows).sort_values(
    ["final_state", "m_mediator", "m_darkphoton", "ctau"]
).reset_index(drop=True)

print(f"{len(signal_table)} signal points; "
      f"yields span {signal_table['yield'].min():.2e} to {signal_table['yield'].max():.2f} "
      f"events at the 1 fb reference cross section")
signal_table.head(15)

In [ ]:
# Signal efficiency landscape: yield vs ctau, one line per (mass, dark photon mass)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

for ax, final_state in zip(axes, ["2Mu2E", "4Mu"]):
    subset = signal_table[signal_table.final_state == final_state]
    for (m_med, m_dp), group in subset.groupby(["m_mediator", "m_darkphoton"]):
        group = group.sort_values("ctau")
        ax.plot(group.ctau, group["yield"], marker="o", ms=4,
                label=f"$m_{{Z_D}}$={m_med:g}, $m_{{d}}$={m_dp:g} GeV")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$c\tau$ [mm]")
    ax.set_ylabel("SR yield at 1 fb [events]")
    ax.set_title(f"{final_state} signal region")
    ax.grid(alpha=0.3)

axes[1].legend(fontsize=6, ncol=2, loc="lower left")
plt.tight_layout()
plt.show()

## 2. Write the datacards

Each card is a one-bin counting experiment: the SR yield of one signal point against the
grouped background yields in the same region.

`observation` is set to the total background, i.e. the cards are blinded and only expected
limits are meaningful from them. `run_combine_limits.py` passes `--run blind` by default, so
Combine builds its Asimov dataset from the background-only expectation and ignores that line
anyway.

Nuisance parameters:

| nuisance | applies to | model |
|---|---|---|
| `lumi_13TeV` | everything | lnN, 2.5% (2018) |
| `mcstat_<channel>_<process>` | each background | `gmN`, from the effective raw MC entries |
| `mcstat_<channel>_signal` | signal | lnN, from the MC statistical error |
| `bkg_norm` | all backgrounds | lnN, **off by default** |

> **The background here is MC, not the ABCD estimate.** The whole point of the ABCD plane is
> that the SR background is predicted from regions B, C and D in *data*; the MC counts used
> here come from one or two raw simulated events and are only a placeholder. That is why the
> per-background nuisance is a `gmN` rather than a log-normal — with one raw event a lognormal
> badly misdescribes the uncertainty, whereas `gmN N alpha` tells Combine the raw count `N` and
> the per-event weight `alpha` and lets it profile the rate with a Gamma distribution.
>
> When the data-driven prediction is available, feed it in as the background rate and set
> `bkg_norm_unc` to its systematic uncertainty (and `mc_stat="lnN"` or `None`).

In [ ]:
config = datacard_tools.DatacardConfig(
    lumi_unc=0.025,      # 2018 integrated luminosity
    mc_stat="gmN",       # "gmN", "lnN", or None
    bkg_norm_unc=None,   # e.g. 0.30 once the ABCD closure uncertainty is known
    signal_unc=None,     # e.g. 0.10 for signal modelling/trigger/ID systematics
    observation="bkg",   # blinded: observation = total background
)

written, floored = datacard_tools.write_datacards(
    signal_yields, bkg_grouped, DATACARD_DIR, config=config,
)

print(f"wrote {len(written)} datacards to {DATACARD_DIR}")
if floored:
    print(f"WARNING: {len(floored)} cards had no non-empty background and were floored "
          f"to {config.min_bkg}: {floored[:5]}")

In [ ]:
# Sanity check one card
example = DATACARD_DIR / "datacard_SR_2mu2e_2Mu2E_1000GeV_1p2GeV_0p96mm.txt"
print(example.read_text())

## 3. Run Combine

Combine lives in its own CMSSW release, so it cannot be imported here — run the script from a
shell:

```bash
cd $CMSSW_BASE/src/SIDM
python sidm/scripts/run_combine_limits.py -j 8
```

Useful options:

* `--combine-cmssw /path/to/CMSSW_X_Y_Z` (or `$COMBINE_CMSSW_BASE`) — the release holding
  `HiggsAnalysis/CombinedLimit`;
* `--unblind` — also compute the observed limit from each card's `observation` line;
* `--pattern 'datacard_SR_4mu_*'` — restrict to one channel;
* `--force` — re-run points already present in `limits.csv` (otherwise they are skipped).

The script picks Combine's `--rMax` per card from that card's S and B, because the faintest
signal points sit at `r` of order 1e6 and would fall outside Combine's default range of 20.

In [ ]:
# Or launch it straight from the notebook
import subprocess

result = subprocess.run(
    [sys.executable, "../../scripts/run_combine_limits.py", "-j", "8"],
    capture_output=True, text=True,
)
print(result.stdout[-3000:])
print(result.stderr[-2000:])

## 4. Read the limits back and plot them

`r` is the limit on the signal cross section in fb, since the signal is normalised to a 1 fb reference.

In [ ]:
limits = pd.read_csv(LIMIT_DIR / "limits.csv")
print(f"{len(limits)} limits")
limits.sort_values("exp").head(10)

In [ ]:
# Expected limit vs ctau, with the +-1 and +-2 sigma bands, for one mass point per panel
def plot_brazil(ax, group, title):
    group = group.sort_values("ctau")
    ax.fill_between(group.ctau, group.exp_m2, group.exp_p2,
                    color="gold", label=r"expected $\pm 2\sigma$")
    ax.fill_between(group.ctau, group.exp_m1, group.exp_p1,
                    color="limegreen", label=r"expected $\pm 1\sigma$")
    ax.plot(group.ctau, group.exp, "k--", label="median expected")
    if "obs" in group and group.obs.notna().any():
        ax.plot(group.ctau, group.obs, "ko-", ms=4, label="observed")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$c\tau$ [mm]")
    ax.set_ylabel(r"95% CL upper limit on $\sigma$ [fb]")
    ax.set_title(title, fontsize=10)
    ax.grid(alpha=0.3)


keys = sorted(limits.groupby(["final_state", "m_mediator", "m_darkphoton"]).groups)
ncols = 4
nrows = int(np.ceil(len(keys) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)

for ax, key in zip(axes.flat, keys):
    final_state, m_med, m_dp = key
    group = limits[(limits.final_state == final_state)
                   & (limits.m_mediator == m_med)
                   & (limits.m_darkphoton == m_dp)]
    plot_brazil(ax, group, f"{final_state}: $m_{{Z_D}}$={m_med:g} GeV, $m_d$={m_dp:g} GeV")

for ax in axes.flat[len(keys):]:
    ax.axis("off")
axes.flat[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# Median expected limit across the full grid, one panel per final state
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

for ax, final_state in zip(axes, ["2Mu2E", "4Mu"]):
    subset = limits[limits.final_state == final_state]
    for (m_med, m_dp), group in subset.groupby(["m_mediator", "m_darkphoton"]):
        group = group.sort_values("ctau")
        ax.plot(group.ctau, group.exp, marker="o", ms=4,
                label=f"$m_{{Z_D}}$={m_med:g}, $m_d$={m_dp:g} GeV")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$c\tau$ [mm]")
    ax.set_ylabel(r"median expected 95% CL limit on $\sigma$ [fb]")
    ax.set_title(f"{final_state}")
    ax.grid(alpha=0.3)

axes[1].legend(fontsize=6, ncol=2)
plt.tight_layout()
plt.savefig(STUDY_DIR / "expected_limits_vs_ctau.png", dpi=150, bbox_inches="tight")
plt.show()